In [1]:
# ============================================================
# inference.py — Standalone BioBERT Inference (Google Colab)
# Run this file independently, no other cells needed
# ============================================================

# ── Mount Drive and install dependencies ──
from google.colab import drive
drive.mount("/content/drive")

import subprocess
subprocess.run(["pip", "install", "transformers==4.40.0", "-q"])

# ── Imports ──
import torch
import numpy as np
import torch.nn as nn
from transformers import BertTokenizer, BertModel

# ── Constants ──
MODEL_NAME   = "dmis-lab/biobert-base-cased-v1.1"
WEIGHTS_PATH = "/content/drive/MyDrive/BioBERT_Project/checkpoints/best_model.pt"
MAX_LENGTH   = 128
TEMPERATURE  = 3
NUM_CLASSES  = 8

DISEASE_NAMES = [
    "Atelectasis",
    "Emphysema",
    "Hiatal Hernia",
    "Pleural Effusion",
    "Pneumonia",
    "Pneumothorax",
    "Pulmonary Congestion",
    "Pulmonary Fibrosis",
]

# ── Model definition (must match training exactly) ──
class BioBERTClassifier(nn.Module):
    def __init__(self, model_name, num_classes=8, dropout_rate=0.3):
        super(BioBERTClassifier, self).__init__()
        self.bert       = BertModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_vector = outputs.last_hidden_state[:, 0, :]
        cls_vector = self.dropout(cls_vector)
        logits     = self.classifier(cls_vector)
        return logits

# ── Load tokenizer ──
print("Loading tokenizer...")
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# ── Load model and weights from Drive ──
print("Loading model architecture...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = BioBERTClassifier(MODEL_NAME, num_classes=NUM_CLASSES)

print(f"Loading fine-tuned weights from Drive...")
checkpoint = torch.load(WEIGHTS_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print(f"Model loaded successfully.")
print(f"  Epoch:        {checkpoint['epoch']}")
print(f"  Val Macro F1: {checkpoint['val_macro_f1']:.4f}")
print(f"  Device:       {device}")
print()

# ── Inference function ──
def predict(text: str) -> dict:
    """
    Takes a symptom sentence.
    Returns calibrated 8-class probability dict for fusion network.
    """
    encoding = tokenizer(
        text,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    input_ids      = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        logits        = model(input_ids, attention_mask)
        scaled_logits = logits / TEMPERATURE
        probs         = torch.softmax(scaled_logits, dim=1)
        probs         = probs.squeeze(0).cpu().numpy()

    return dict(zip(DISEASE_NAMES, probs))


# ── Run test inference ──
print("=" * 50)
print("TEST INFERENCE")
print("=" * 50)

test_sentences = [
    "The patient presents with shortness of breath, fever, cough, and chest pain.",
    "The patient presents with nausea, back pain, burning abdominal pain, and heartburn.",
    "The patient presents with sharp chest pain, drug abuse, and shoulder pain.",
]

for text in test_sentences:
    result = predict(text)
    print(f"\nInput: {text}")
    print(f"Output:")
    for disease, prob in result.items():
        bar = "█" * int(prob * 30)
        print(f"  {disease:<25} {prob:.4f}  {bar}")
    print(f"  Sum: {sum(result.values()):.6f}")
    print(f"  Predicted: {max(result, key=result.get)}")

Mounted at /content/drive
Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Loading model architecture...


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading fine-tuned weights from Drive...
Model loaded successfully.
  Epoch:        4
  Val Macro F1: 0.9579
  Device:       cpu

TEST INFERENCE

Input: The patient presents with shortness of breath, fever, cough, and chest pain.
Output:
  Atelectasis               0.0482  █
  Emphysema                 0.0664  █
  Hiatal Hernia             0.0423  █
  Pleural Effusion          0.0357  █
  Pneumonia                 0.0702  ██
  Pneumothorax              0.0428  █
  Pulmonary Congestion      0.0750  ██
  Pulmonary Fibrosis        0.6194  ██████████████████
  Sum: 1.000000
  Predicted: Pulmonary Fibrosis

Input: The patient presents with nausea, back pain, burning abdominal pain, and heartburn.
Output:
  Atelectasis               0.0293  
  Emphysema                 0.0397  █
  Hiatal Hernia             0.7812  ███████████████████████
  Pleural Effusion          0.0305  
  Pneumonia                 0.0270  
  Pneumothorax              0.0250  
  Pulmonary Congestion      0.0357  █
  Pulmo